<a href="https://colab.research.google.com/github/lubkabubka/-/blob/main/%D0%A1%D1%80%D0%B0%D0%B2%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5_%D0%BC%D0%B5%D1%82%D0%BE%D0%B4%D0%BE%D0%B2_%D1%82%D0%B5%D0%BC%D0%B0%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%BE%D0%B9_%D0%BA%D0%BB%D0%B0%D1%81%D1%81%D0%B8%D1%84%D0%B8%D0%BA%D0%B0%D1%86%D0%B8%D0%B8_%D1%82%D0%B5%D0%BA%D1%81%D1%82%D0%BE%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import random
import warnings
from collections import Counter
from typing import List, Tuple

import numpy as np
import torch
from torch import nn, optim
from tqdm.auto import tqdm, trange

import kagglehub
import pandas as pd
import os
from datasets import load_dataset

from collections import Counter
import re

warnings.filterwarnings("ignore")

In [4]:
def clean_text(text):
    #привести к нижнему виде
    text = text.lower()
    # убрать html-артефакты типа &quot; &#39; и т.п.
    text = re.sub(r"&\w+;|#\d+;|&#\d+;", " ", text)
    # заменить все, кроме букв, цифр и пробелов, на пробел
    text = re.sub(r"[^a-zа-яё0-9\s]", " ", text)
    # убрать лишние пробелы
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [60]:
dataset_name = 2
# 1 — AG News
# 2 — DBpedia
# 3 — RuNews

if dataset_name == 1:
    # AG News
    path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")

    print("Path to dataset files:", path)

    train_df = pd.read_csv(os.path.join(path, "train.csv"))
    test_df = pd.read_csv(os.path.join(path, "test.csv"))

    train_df["text"] = train_df["Title"].astype(str) + " " + train_df["Description"].astype(str)
    test_df["text"] = test_df["Title"].astype(str) + " " + test_df["Description"].astype(str)

    train_df["Class Index"] = train_df["Class Index"] - 1
    test_df["Class Index"] = test_df["Class Index"] - 1

elif dataset_name == 2:
    # DBpedia
    dbpedia = load_dataset("dbpedia_14")

    train_df = pd.DataFrame(dbpedia["train"])
    test_df = pd.DataFrame(dbpedia["test"])

    train_df["Class Index"] = train_df["label"]
    test_df["Class Index"] = test_df["label"]

    train_df["text"] = train_df["title"].astype(str) + " " + train_df["content"].astype(str)
    test_df["text"] = test_df["title"].astype(str) + " " + test_df["content"].astype(str)

elif dataset_name == 3:
    # RuNews
    runews = load_dataset("data-silence/rus_news_classifier")
    train_df = pd.DataFrame(runews["train"])
    test_df = pd.DataFrame(runews["test"])

    train_df["Class Index"] = train_df["labels"]
    test_df["Class Index"] = test_df["labels"]

    train_df["text"] = train_df["news"].astype(str)
    test_df["text"] = test_df["news"].astype(str)

else:
    raise ValueError("dataset_name должен быть 1, 2 или 3")


#оставили только нужные столбцы
train_df = train_df[["Class Index", "text"]]
test_df = test_df[["Class Index", "text"]]

#очистили текст
train_df["text"] = train_df["text"].astype(str).apply(clean_text)
test_df["text"] = test_df["text"].astype(str).apply(clean_text)

print(train_df.head())
print(test_df.head())

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Classes:", sorted(train_df["Class Index"].unique()))

README.md: 0.00B [00:00, ?B/s]

dbpedia_14/train-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

dbpedia_14/test-00000-of-00001.parquet:   0%|          | 0.00/13.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/70000 [00:00<?, ? examples/s]

   Class Index                                               text
0            0  e d abbott ltd abbott of farnham e d abbott li...
1            0  schwan stabilo schwan stabilo is a german make...
2            0  q workshop q workshop is a polish company loca...
3            0  marvell software solutions israel marvell soft...
4            0  bergan mercy medical center bergan mercy medic...
   Class Index                                               text
0            0  ty ku ty ku ta ku is an american alcoholic bev...
1            0  odd lot entertainment oddlot entertainment fou...
2            0  henkel henkel ag company kgaa operates worldwi...
3            0  goat store the goat store games of all type st...
4            0  ragwing aircraft designs ragwing aircraft desi...
Train shape: (560000, 2)
Test shape: (70000, 2)
Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int

Word2Vec

In [61]:
text = " ".join(train_df["text"].tolist())
text = clean_text(text)
words = text.split()
word_counts = Counter(words) #формирует словарь
words = [w for w in words if word_counts[w] > 5] #удаляем малочастотные слова

In [62]:
print(f"Beginning of the text: {words[:30]}")
print(f"Total words in text: {len(words)}")
print(f"Unique words: {len(set(words))}")

Beginning of the text: ['e', 'd', 'abbott', 'ltd', 'abbott', 'of', 'farnham', 'e', 'd', 'abbott', 'limited', 'was', 'a', 'british', 'coachbuilding', 'business', 'based', 'in', 'farnham', 'surrey', 'trading', 'under', 'that', 'name', 'from', '1929', 'a', 'major', 'part', 'of']
Total words in text: 27456838
Unique words: 119964


In [63]:
!wget https://raw.githubusercontent.com/hse-ds/iad-deep-learning/master/2022/seminars/sem08/utils.py

--2026-05-03 07:19:59--  https://raw.githubusercontent.com/hse-ds/iad-deep-learning/master/2022/seminars/sem08/utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1511 (1.5K) [text/plain]
Saving to: ‘utils.py.1’

utils.py.1          100%[===================>]   1.48K  --.-KB/s    in 0s      

2026-05-03 07:19:59 (61.4 MB/s) - ‘utils.py.1’ saved [1511/1511]



In [64]:
import utils
#Два словаря для преобразования слов в целые числа и обратно.
#Целые числа присваиваются в порядке убывания частоты (самому частому слову - 0 , следующему по частоте —  1 и так далее).
vocab_to_int, int_to_vocab = utils.create_lookup_tables(words)
int_words = [vocab_to_int[word] for word in words] #преобразуем слова в цифры

print(int_words[:30])
print(len(int_words))

[70, 81, 7243, 762, 7243, 2, 14709, 70, 81, 7243, 552, 6, 3, 112, 65169, 336, 63, 1, 14709, 3820, 2596, 140, 36, 74, 15, 1229, 3, 333, 86, 2]
27456838


#### Проводим субдискретизацию (subsampling)
Часто встречающиеся слова (например "the", "of", "for", etc.) не обеспечивают особого контекста для близлежащих слов. Если мы отбросим некоторые из них, мы сможем удалить часть шума из наших данных и взамен получить более быстрое обучение и лучшее представление. Для каждого слова $w_i$ в обучающем наборе мы отбрасываем его с вероятностью, равной

$$ P(w_i) = 1 - \sqrt{\frac{t}{f(w_i)}}, $$

где $t$ - пороговый параметр, а $f(w_i)$ - частота слова $w_i$ в общем наборе данных.

In [65]:
threshold = 1e-5
word_counts = Counter(int_words)  #словарь слово - количество раз, которое оно в тексте встречается
print(f"42-th word appears in the text {word_counts[42]} times")

train_words = []

for w in int_words:
    if random.random() < 1 - (threshold / (word_counts[w] / len(int_words)))**0.5:
      continue
    else:
      train_words.append(w)

print(int_words[:30])
print(train_words[:30])
print(len(train_words))
print(len(set(train_words)))

42-th word appears in the text 51487 times
[70, 81, 7243, 762, 7243, 2, 14709, 70, 81, 7243, 552, 6, 3, 112, 65169, 336, 63, 1, 14709, 3820, 2596, 140, 36, 74, 15, 1229, 3, 333, 86, 2]
[7243, 762, 7243, 14709, 70, 7243, 112, 65169, 14709, 333, 7563, 1716, 4358, 1739, 611, 86215, 86215, 3792, 20950, 36357, 11866, 30034, 20950, 86215, 6426, 6568, 6568, 978, 2273, 439]
7916477
119964


Word2Vec Skip-gram

In [66]:
#выдает контекст слова на позиции idx
def get_target(words: List[int], idx: int, window_size: int = 5) -> List[int]:
    target = []
    window = random.randint(1, window_size)

    start = max(0, idx - window)
    end = min(len(words), idx + window + 1)

    for i in range(start, end):
        if i != idx:
            target.append(words[i])

    return target

In [67]:
#Тест кода
int_text = [i for i in range(10)]
idx = 5
target = get_target(int_text, idx=idx, window_size=5)
print("Input: ", int_text)
print(f"Index of interest: {idx}")
print("Target: ", target)

Input:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Index of interest: 5
Target:  [0, 1, 2, 3, 4, 6, 7, 8, 9]


In [68]:
#выдает батчи
def get_batches(words: List[int], batch_size: int, window_size: int = 5) -> Tuple[List[int], List[int]]:
    for i in range(0, len(words) - len(words) % batch_size, batch_size):
        x, y = [], []
        batch = words[i : i + batch_size]
        for j in range(len(batch)):
            batch_x = batch[j]
            batch_y = get_target(words, i + j, window_size)
            y.extend(batch_y)
            x.extend([batch_x] * len(batch_y))
        yield x, y

In [69]:
#Тест кода

int_text = [5233, 3080, 11, 5, 194, 1, 3133, 45, 58]
batch_gen = get_batches(int_text, batch_size=4, window_size=5)
count = len(int_text) // 4
for i in range(count):
  x, y = next(batch_gen)
  print(f"x: {x}")
  print(f"y: {y}\n")

x: [5233, 5233, 5233, 5233, 5233, 3080, 3080, 3080, 3080, 3080, 3080, 11, 11, 5, 5, 5, 5, 5, 5]
y: [3080, 11, 5, 194, 1, 5233, 11, 5, 194, 1, 3133, 3080, 5, 5233, 3080, 11, 194, 1, 3133]

x: [194, 194, 194, 194, 194, 194, 194, 194, 1, 1, 1, 1, 1, 1, 1, 1, 3133, 3133, 3133, 3133, 45, 45, 45]
y: [5233, 3080, 11, 5, 1, 3133, 45, 58, 5233, 3080, 11, 5, 194, 3133, 45, 58, 194, 1, 45, 58, 1, 3133, 58]



In [70]:
def cosine_similarity(
    embedding: nn.Module,
    valid_size: int = 16,
    valid_window: int = 100,
    device: str = "cpu",
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Функция используется для проверки качества обученных эмбеддингов.

    Она выбирает несколько случайных слов из словаря и считает,
    насколько они похожи на остальные слова по косинусной близости.

    Если эмбеддинги обучились хорошо, то у похожих по смыслу слов
    должны быть близкие векторы. Например, рядом со словом "sport"
    могут оказаться "football", "game", "team".

    :param embedding: обученный слой nn.Embedding, в котором хранятся векторы слов
    :param valid_size: сколько слов выбрать для проверки
    :param valid_window: из какого диапазона частых слов выбирать примеры
    :param device: устройство для вычислений: "cpu" или "cuda"
    :return: индексы выбранных слов и матрица похожестей этих слов со всеми словами словаря
    """

    # Берем матрицу эмбеддингов.
    # Каждая строка этой матрицы — вектор одного слова.
    embed_vectors = embedding.weight

    # Считаем длину каждого вектора из матрицы эмбеддингов.
    # Это нужно для формулы косинусной близости:
    # cos(a, b) = (a · b) / (|a| |b|)
    magnitudes = embed_vectors.pow(2).sum(dim=1).sqrt().unsqueeze(0)

    # Выбираем несколько слов для проверки.
    # Половину берем из самых частых слов: индексы от 0 до valid_window.
    # Чем меньше индекс слова, тем чаще оно встречается в корпусе.
    valid_examples = np.array(random.sample(range(valid_window), valid_size // 2))

    # Вторую половину берем из слов с индексами от 1000 до 1000 + valid_window.
    # Это уже не самые частые слова, но они тоже достаточно часто встречаются.
    valid_examples = np.append(
        valid_examples, random.sample(range(1000, 1000 + valid_window), valid_size // 2)
    )

    # Переводим индексы выбранных слов в тензор PyTorch
    # и отправляем его на нужное устройство.
    valid_examples = torch.LongTensor(valid_examples).to(device)

    # Получаем векторы выбранных слов.
    valid_vectors = embedding(valid_examples)

    # Считаем похожесть выбранных слов со всеми словами из словаря.
    # Результат — матрица размера:
    # количество выбранных слов × размер словаря.
    #
    # В каждой строке лежат похожести одного выбранного слова
    # со всеми словами из словаря.
    similarities = torch.mm(valid_vectors, embed_vectors.t()) / magnitudes

    # Возвращаем:
    # valid_examples — индексы слов, которые проверяли;
    # similarities — похожести этих слов со всеми словами словаря.
    return valid_examples, similarities

In [71]:
class SkipGram(nn.Module):
    def __init__(self, vocab_size: int, embed_size: int):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.linear = nn.Linear(embed_size, vocab_size)

    def forward(self, x: torch.Tensor):
        out = self.linear(self.embed(x))
        return out

In [72]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
print_every = 1000
steps = 0
n_epochs = 5
batch_size = 1024
embedding_dim = 128

model = SkipGram(len(vocab_to_int), embedding_dim)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

for e in trange(n_epochs, leave=True, desc="Epoch number"):
    pbar = tqdm(
        get_batches(train_words, batch_size),
        leave=False,
        desc="Batch number",
        total=len(train_words) // batch_size,
    )

    # get input and target batches
    for inputs, targets in pbar:
        steps += 1
        inputs, targets = torch.LongTensor(inputs), torch.LongTensor(targets)
        inputs, targets = inputs.to(device), targets.to(device)

        log_ps = model(inputs)
        loss = criterion(log_ps, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if steps % print_every == 0:
            # getting examples and similarities
            valid_examples, valid_similarities = cosine_similarity(
                model.embed, device=device
            )
            _, closest_idxs = valid_similarities.topk(6)

            valid_examples, closest_idxs = valid_examples.to("cpu"), closest_idxs.to(
                "cpu"
            )
            for ii, valid_idx in enumerate(valid_examples):
                closest_words = [int_to_vocab[idx.item()] for idx in closest_idxs[ii]][
                    1:
                ]
                print(int_to_vocab[valid_idx.item()] + " | " + ", ".join(closest_words))
            print("...")

cuda


Epoch number:   0%|          | 0/5 [00:00<?, ?it/s]

Batch number:   0%|          | 0/7730 [00:00<?, ?it/s]

south | бюро, falcate, elsinore, homolje, mindless
on | y, boost, tseung, laoghaire, controllers
are | zagora, trichosalpinx, patient, letouzeyi, silverwater
county | roberts, wenham, frisco, 746, sandvika
as | liberal, conformed, prorella, kindergarten, haris
born | gine, ues, vashi, sarnoff, unorthodox
district | unwitting, esmond, 53000, toyonaka, duabanga
lake | cashmore, optics, ringier, morra, malas
silent | etty, chantiers, spanning, olinda, drumma
fire | lourie, berthing, paracles, floridan, digyna
fc | casualty, dineh, prudnicka, dettwiler, hester
don | ampere, chauvenet, homalopoma, mondal, dominatrix
jean | skram, ansellia, kitamura, hangars, mission
cathedral | bielany, ulmus, roza, zamiaceae, cadena
self | chaharbagh, nur, slavonski, crypt, aspiration
committee | rf, pump, zimbalist, analogue, legendary
...
college | francis, 1850, coton, eriogonum, caucasian
she | anguillan, wolves, gaps, sujata, kids
of | hoist, helwan, pressings, pornographic, hathor
east | antonella, r

Batch number:   0%|          | 0/7730 [00:00<?, ?it/s]

...

state | national, states, currently, of, indian
by | is, in, a, was, the
new | york, australia, zealand, in, melbourne
d | b, e, l, f, informatique
she | her, herself, married, his, who
to | the, of, and, that, with
family | species, found, endemic, loss, common
in | the, it, a, is, and
fort | worth, meat, florida, lakes, kootenai
basketball | soccer, player, alicja, fenella, escalating
better | stage, example, interest, ghazaleh, speed
locomotive | railways, freight, locomotives, steam, trainload
close | border, ladislao, crocodile, trifurcula, foes
lawyer | law, governor, workbooks, pyroderces, researcher
founder | entrepreneur, seiwa, tetraodontidae, chiharu, brand
fire | uniondale, cannibalistic, punchline, fake, daf
...
was | an, in, and, a, the
college | university, established, state, private, located
located | county, high, district, school, it
that | to, what, well, other, this
his | he, and, who, as, work
historic | providence, azusa, register, pennsylvania, somerton
are | it

Batch number:   0%|          | 0/7730 [00:00<?, ?it/s]

first | originally, was, by, it, second
known | as, also, is, called, in
east | west, north, km, south, in
family | plant, found, species, loss, endemic
2006 | 2012, 2011, was, province, iran
species | genus, endemic, native, flowering, threatened
one | of, the, in, it, two
an | a, and, is, as, the
reservoir | dam, water, hydroelectric, dammed, calaveras
nepal | kathmandu, nepali, bhutan, 1991, zone
horror | thriller, supernatural, author, novel, written
snails | mollusks, mollusk, gastropod, snail, austrodaphnella
medium | sized, large, punjab, meerut, belongs
representative | indigenous, balu, markets, segal, aea
1951 | 1949, 1968, 1956, 1955, 1952
command | commander, comdr, military, lt, naval
...
college | university, established, undergraduate, located, community
by | is, the, a, 1998, under
north | east, west, south, located, km
district | county, rural, public, located, central
world | s, war, the, one, among
n | l, y, p, h, spanish
which | the, on, with, been, to
has | been, w

Batch number:   0%|          | 0/7730 [00:00<?, ?it/s]

is | a, in, and, the, it
a | is, and, in, the, of
published | by, first, was, it, author
album | studio, solo, singer, released, songwriter
on | and, the, it, with, october
be | not, however, can, would, confused
film | directed, drama, starring, comedy, starred
species | genus, endemic, plant, native, flowering
representative | serially, детектив, balu, wading, ephestia
silent | drama, starring, directed, 1921, 1924
spain | spanish, barcelona, madrid, italy, iberian
introduced | continents, eurasia, elsewhere, noxious, parts
pass | mountain, pedestrian, summit, route, trail
scott | orson, card, welles, richard, thompson
battle | battles, fought, war, squadron, siege
jones | sam, stone, jonathan, philly, 4
...
new | york, zealand, jersey, a, city
was | in, by, the, it, a
been | since, has, have, it, with
who | father, parents, his, whose, old
company | inc, owned, subsidiary, corporation, production
part | the, in, usa, is, located
de | la, spanish, french, le, l
the | in, it, is, of, 

In [73]:
import torch
import torch.nn.functional as F

def get_word_vector(word, model, vocab_to_int, device="cpu"):
    if word not in vocab_to_int:
        raise ValueError(f"Слова '{word}' нет в словаре")

    idx = torch.tensor([vocab_to_int[word]], dtype=torch.long).to(device)
    vector = model.embed(idx).detach().cpu()
    return vector


def most_similar_words(word, model, vocab_to_int, int_to_vocab, top_k=10, device="cpu"):
    if word not in vocab_to_int:
        print(f"Слова '{word}' нет в словаре")
        return

    # матрица эмбеддингов всех слов
    embeddings = model.embed.weight.detach()  # [vocab_size, emb_dim]

    # вектор нужного слова
    word_idx = vocab_to_int[word]
    word_vector = embeddings[word_idx].unsqueeze(0)  # [1, emb_dim]

    # косинусные близости ко всем словам
    similarities = F.cosine_similarity(word_vector, embeddings, dim=1)

    # top_k + 1, потому что само слово тоже попадет
    values, indices = torch.topk(similarities, top_k + 1)

    result = []
    for idx, sim in zip(indices.cpu().numpy(), values.cpu().numpy()):
        similar_word = int_to_vocab[idx]
        if similar_word != word:
            result.append((similar_word, float(sim)))
        if len(result) == top_k:
            break

    return result


def word_similarity(word1, word2, model, vocab_to_int, device="cpu"):
    if word1 not in vocab_to_int:
        print(f"Слова '{word1}' нет в словаре")
        return
    if word2 not in vocab_to_int:
        print(f"Слова '{word2}' нет в словаре")
        return

    vec1 = get_word_vector(word1, model, vocab_to_int, device)
    vec2 = get_word_vector(word2, model, vocab_to_int, device)

    sim = F.cosine_similarity(vec1, vec2).item()
    return sim

In [74]:
print(most_similar_words("president", model, vocab_to_int, int_to_vocab, top_k=10, device=device))
print(most_similar_words("game", model, vocab_to_int, int_to_vocab, top_k=10, device=device))
print(most_similar_words("market", model, vocab_to_int, int_to_vocab, top_k=10, device=device))

[('presidential', 0.5907073020935059), ('secretary', 0.5789197683334351), ('presidents', 0.5660525560379028), ('presidency', 0.5567872524261475), ('barack', 0.5139314532279968), ('obama', 0.5082910060882568), ('served', 0.49969229102134705), ('assassination', 0.49459293484687805), ('minister', 0.49372339248657227), ('vice', 0.4932827949523926)]
[('games', 0.6256504058837891), ('playing', 0.5704519152641296), ('video', 0.5639747977256775), ('dungeons', 0.5224844813346863), ('role', 0.5102905035018921), ('gaming', 0.505242109298706), ('product', 0.501571774482727), ('advanced', 0.5011310577392578), ('xbox', 0.48998522758483887), ('dragons', 0.4717625677585602)]
[('companies', 0.47948434948921204), ('markets', 0.4649181365966797), ('sales', 0.4598396420478821), ('became', 0.4543731212615967), ('later', 0.44793421030044556), ('financial', 0.44430702924728394), ('value', 0.43811851739883423), ('retail', 0.4359513819217682), ('sold', 0.4298633933067322), ('securities', 0.42228198051452637)]


In [75]:
#словарь с эмбендингами
embedding_dict_skip_gram = {
    word: model.embed.weight[idx].detach().cpu().numpy()
    for word, idx in vocab_to_int.items()
}

Перейдем к задаче самой классификации.

In [76]:
import numpy as np

#Каждый документ будет характеризовать 1 вектор - усреднение эмбендингов всех слов из текста

def get_document_embedding(text, embedding_dict, embedding_dim=128):
    words = text.split()
    vectors = [embedding_dict[word] for word in words if word in embedding_dict]

    if len(vectors) == 0:
        return np.zeros(embedding_dim)

    return np.mean(vectors, axis=0)

In [77]:
#получаем массив вектор, каждый из которых характеризует текст
X_train_emb = np.array([
    get_document_embedding(text, embedding_dict_skip_gram, embedding_dim=128)
    for text in train_df["text"]
])

X_test_emb = np.array([
    get_document_embedding(text, embedding_dict_skip_gram, embedding_dim=128)
    for text in test_df["text"]
])

In [78]:
y_train = train_df["Class Index"].values
y_test = test_df["Class Index"].values

In [79]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_emb, y_train)

y_pred = clf.predict(X_test_emb)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print(classification_report(y_test, y_pred))

Accuracy: 0.9366428571428571
Macro F1: 0.9365106497573794
              precision    recall  f1-score   support

           0       0.88      0.86      0.87      5000
           1       0.95      0.95      0.95      5000
           2       0.85      0.83      0.84      5000
           3       0.97      0.97      0.97      5000
           4       0.94      0.94      0.94      5000
           5       0.96      0.97      0.96      5000
           6       0.93      0.93      0.93      5000
           7       0.97      0.98      0.97      5000
           8       0.99      0.98      0.98      5000
           9       0.98      0.98      0.98      5000
          10       0.98      0.98      0.98      5000
          11       0.93      0.95      0.94      5000
          12       0.93      0.94      0.94      5000
          13       0.86      0.87      0.86      5000

    accuracy                           0.94     70000
   macro avg       0.94      0.94      0.94     70000
weighted avg       0.9

Word2Vec CBOW

In [80]:
#выдает контекст слова
def get_context(words, idx, window_size=2):
    start = max(0, idx - window_size)
    end = min(len(words), idx + window_size + 1)

    context = [words[i] for i in range(start, end) if i != idx]
    return context

In [81]:
#генерирует батчи
def get_batches_cbow(words: List[int], batch_size: int, window_size: int = 2) -> Tuple[List[List[int]], List[int]]:
    for i in range(
        window_size,
        len(words) - window_size - (len(words) - 2 * window_size) % batch_size,
        batch_size,
    ):
        x, y = [], []
        batch = words[i : i + batch_size]

        for j in range(len(batch)):
            idx = i + j
            x.append(get_context(words, idx, window_size))
            y.append(words[idx])

        yield x, y

In [82]:
#Тест кода
int_text = [5233, 3080, 11, 5, 194, 1, 3133, 45, 58]
batch_gen = get_batches_cbow(int_text, batch_size=4, window_size=2)
for x, y in batch_gen:
    print(f"x: {x}")
    print(f"y: {y}\n")

x: [[5233, 3080, 5, 194], [3080, 11, 194, 1], [11, 5, 1, 3133], [5, 194, 3133, 45]]
y: [11, 5, 194, 1]



In [83]:
  class CBOW(nn.Module):
    def __init__(self, vocab_size: int, embed_size: int):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.linear = nn.Linear(embed_size, vocab_size)

    def forward(self, x: torch.Tensor):
        out = self.linear(self.embed(x).mean(dim=1))
        return out

In [84]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
print_every = 1000
steps = 0
n_epochs = 5
batch_size = 1024
embedding_dim = 128

model = CBOW(len(vocab_to_int), embedding_dim)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

window_size = 2
valid_size = len(train_words) - 2 * window_size
total_batches = valid_size // batch_size

for e in trange(n_epochs, leave=True, desc="Epoch number"):
    pbar = tqdm(
      get_batches_cbow(train_words, batch_size, window_size),
      leave=False,
      desc="Batch number",
      total=total_batches,
    )

    # get input and target batches
    for inputs, targets in pbar:
        steps += 1
        inputs, targets = torch.LongTensor(inputs), torch.LongTensor(targets)
        inputs, targets = inputs.to(device), targets.to(device)

        log_ps = model(inputs)
        loss = criterion(log_ps, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if steps % print_every == 0:
            # getting examples and similarities
            valid_examples, valid_similarities = cosine_similarity(
                model.embed, device=device
            )
            _, closest_idxs = valid_similarities.topk(6)

            valid_examples, closest_idxs = valid_examples.to("cpu"), closest_idxs.to(
                "cpu"
            )
            for ii, valid_idx in enumerate(valid_examples):
                closest_words = [int_to_vocab[idx.item()] for idx in closest_idxs[ii]][
                    1:
                ]
                print(int_to_vocab[valid_idx.item()] + " | " + ", ".join(closest_words))
            print("...")

cuda


Epoch number:   0%|          | 0/5 [00:00<?, ?it/s]

Batch number:   0%|          | 0/7730 [00:00<?, ?it/s]

first | nazz, motors, tuileries, northampton, hepatology
two | cotham, evren, orofino, slk, parasitizes
or | 460000, 2590, эдуард, bei, annunzio
province | gunundaal, pitlochry, 1936a, karateka, gibson
city | bolca, stylosa, kaohsiung, sarawak, bruiser
historic | conor, citron, lovitt, warblers, wardwell
are | elizabethton, grunau, eyd, sponsored, dasyphyllum
known | slop, clacton, pyjama, steno, sailed
went | governors, mi, reto, brazzaville, kiskiminetas
lives | covered, debonair, mallya, sophora, chb
fc | oakhurst, sian, rodionova, laevigata, 12500
soviet | riparian, quartets, medals, oblates, gokak
tennessee | uab, stagger, loche, dict, refineries
available | panamanian, translate, amoreuxia, ichijinsha, 1890s
scott | kuntur, exhaustive, zhizn, autonomist, eurydera
far | hoeven, arthunkal, galearctus, unwitting, spathulate
...
two | cotham, slk, orofino, margetson, evren
n | hitch, porthcawl, warthog, nazarabad, emamabad
were | silistra, disarm, 1060, hildesheim, kingsway
she | pan

Batch number:   0%|          | 0/7730 [00:00<?, ?it/s]

...

district | county, rural, moga, central, shockwave
lake | lakes, 2740, impoundment, westerlo, separating
central | east, district, province, poland, phanerophlebiopsis
as | known, ilx, pricing, also, hitachi
north | south, west, east, continent, kilometres
national | gautama, culture, wildlife, priklyucheniya, bebek
are | zv, majority, purchases, those, nearly
based | headquartered, bofi, york, lubawskie, scorcher
previous | lhus, heavier, hammill, usual, einkorn
spain | spanish, tenerife, zaragoza, pyrenees, buenos
notable | references, significant, descriptions, frequently, own
bird | birds, kesgrave, minted, bucerotidae, laniarius
locomotive | locomotives, diesel, electric, wagr, nec
prominent | magnet, scholar, strongly, reflections, hup
subsidiary | owned, mitsui, matterhorn, corporation, brand
information | behaviour, encouraging, emphasis, kloss, serials
...
of | the, hacettepe, helseth, mentioned, oberlin
de | la, esp, rio, french, catalunya
released | records, release, label, 

Batch number:   0%|          | 0/7730 [00:00<?, ?it/s]

music | record, studio, label, contemporary, electronic
historic | places, register, carbarn, cullman, tamilnadu
university | law, slavonic, campus, graduated, sciences
american | an, plaster, bumpass, specializing, hubbell
mi | km, kilometres, lies, poland, voivodeship
the | in, of, 901, s, inquiry
one | among, three, four, finest, times
his | him, who, himself, her, lengthy
nepal | zone, time, solan, digicel, traumatized
foundation | decorations, reestablished, seenu, godbout, niches
snails | incised, endfield, fraction, claws, mitrovi
countries | across, 1300, tocantins, giulia, americas
bill | glynn, bruford, martha, helfer, spoofs
affiliated | state, church, university, developmental, osmania
jean | baptiste, michel, desjardins, berens, julien
however | but, never, abandoned, accept, approves
...
found | family, native, mediterranean, species, neotropics
it | awarding, in, medya, bankers, killam
part | dreamworks, outreach, pern, unincorporated, county
house | grist, residence, re

Batch number:   0%|          | 0/7730 [00:00<?, ?it/s]

new | york, brooklyn, sydney, city, tinton
that | what, process, conducive, acquire, eupodium
states | united, america, midwestern, usa, intermountain
family | plant, genus, species, plants, found
house | grist, outbuilding, residence, waltham, mansion
d | lebaudy, m, z, tudes, kohlmann
its | but, population, 2006, at, existence
n | jaane, mccann, ngti, nf, nguy
information | communicates, epistemology, contacts, behaviour, discrete
technical | szewce, instrumentation, purposes, eradication, oems
subsidiary | owned, chinle, brand, wholly, inc
jones | seaman, bountiful, maccoll, molloy, shadrach
awards | award, won, nominations, 55th, nominated
far | latitudes, thebit, mokowanis, lies, xanthi
however | but, after, never, him, although
scott | sheen, ralph, costner, eason, reesor
...
west | north, east, within, km, cabarrus
e | sorkh, persian, swuf, l, begg
was | by, it, february, in, october
river | tributary, headwaters, tributaries, essequibo, stream
city | in, located, is, new, metro

In [85]:
print(most_similar_words("president", model, vocab_to_int, int_to_vocab, top_k=10, device=device))
print(most_similar_words("game", model, vocab_to_int, int_to_vocab, top_k=10, device=device))
print(most_similar_words("market", model, vocab_to_int, int_to_vocab, top_k=10, device=device))

[('presidents', 0.4744245409965515), ('presidency', 0.46850311756134033), ('adviser', 0.4386330842971802), ('presidential', 0.4272781014442444), ('assassination', 0.4138002097606659), ('impeachment', 0.41069647669792175), ('secretary', 0.4015433192253113), ('recounted', 0.3899042010307312), ('appointed', 0.378660649061203), ('cabinet', 0.37606844305992126)]
[('games', 0.5445411205291748), ('ubisoft', 0.3883959650993347), ('nintendo', 0.3652731478214264), ('soundtracks', 0.3624071478843689), ('dramaturgy', 0.35777735710144043), ('developer', 0.353920578956604), ('foxxx', 0.353800505399704), ('aic', 0.3517044484615326), ('nobita', 0.3503481447696686), ('gaming', 0.3476513624191284)]
[('livelihood', 0.3705766201019287), ('leisure', 0.3608751893043518), ('unsustainable', 0.3534383177757263), ('thunderbird', 0.3451519012451172), ('gone', 0.3420860767364502), ('insures', 0.3406646251678467), ('tenant', 0.3402777314186096), ('bespoke', 0.33777904510498047), ('lvmh', 0.3347543776035309), ('dar

In [86]:
#словарь эмбендингов для cbow
embedding_dict_cbow_gram = {
    word: model.embed.weight[idx].detach().cpu().numpy()
    for word, idx in vocab_to_int.items()
}

In [87]:
import numpy as np

def get_document_embedding(text, embedding_dict, embedding_dim=128):
    words = text.split()
    vectors = [embedding_dict[word] for word in words if word in embedding_dict]

    if len(vectors) == 0:
        return np.zeros(embedding_dim)

    return np.mean(vectors, axis=0)

In [88]:
X_train_emb = np.array([
    get_document_embedding(text, embedding_dict_cbow_gram, embedding_dim=128)
    for text in train_df["text"]
])

X_test_emb = np.array([
    get_document_embedding(text, embedding_dict_cbow_gram, embedding_dim=128)
    for text in test_df["text"]
])

In [89]:
y_train = train_df["Class Index"].values
y_test = test_df["Class Index"].values

In [90]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_emb, y_train)

y_pred = clf.predict(X_test_emb)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print(classification_report(y_test, y_pred))

Accuracy: 0.9449571428571428
Macro F1: 0.9448940970131128
              precision    recall  f1-score   support

           0       0.89      0.87      0.88      5000
           1       0.95      0.97      0.96      5000
           2       0.88      0.86      0.87      5000
           3       0.98      0.97      0.97      5000
           4       0.95      0.95      0.95      5000
           5       0.97      0.97      0.97      5000
           6       0.93      0.93      0.93      5000
           7       0.97      0.98      0.97      5000
           8       0.99      0.98      0.99      5000
           9       0.98      0.98      0.98      5000
          10       0.98      0.98      0.98      5000
          11       0.95      0.96      0.96      5000
          12       0.94      0.95      0.94      5000
          13       0.87      0.88      0.88      5000

    accuracy                           0.94     70000
   macro avg       0.94      0.94      0.94     70000
weighted avg       0.9

Bag of Words

In [91]:
text = " ".join(train_df["text"].tolist())
words = text.split()
word_counts = Counter(words)
words = [w for w in words if word_counts[w] > 5] #оставила только слова которые больше 5 раз встречаются
dic = set(words)
print(list(dic)[:30])

['gas', 'lessons', 'shrew', 'mummies', 'superfluous', 'crawfordsville', 'uprija', 'harmonies', 'possum', 'tangra', 'rimmer', 'holidays', 'seis', 'literatures', '1566', 'sanj', 'melvill', 'maggot', 'ovatus', 'narnia', 'solid', 'caperton', 'barquement', 'dealey', 'jumbo', 'stevensonii', 'parasuram', 'umbrella', 'ollie', 'seibel']


In [92]:
def keep_only_dict_words(text):
    words = text.split()
    words = [w for w in words if w in dic]
    return " ".join(words)

train_df["text"] = train_df["text"].apply(keep_only_dict_words)

In [93]:
dic = list(dic)
dic.append("UNK")
print(len(dic))

119965


In [94]:
count = 0
d = dict()
for i in range(len(dic)):
  d[dic[i]] = count
  count += 1

In [95]:
print(d["UNK"])

119964


In [96]:
train_df["text"][0]

'e d abbott ltd abbott of farnham e d abbott limited was a british coachbuilding business based in farnham surrey trading under that name from 1929 a major part of their output was under sub contract to motor vehicle manufacturers their business closed in 1972'

In [97]:
import numpy as np
from scipy.sparse import lil_matrix

X_train = lil_matrix((len(train_df), len(d)), dtype=np.int32)

for i in range(len(train_df)):
    for w in train_df["text"].iloc[i].split():
        idx = d[w] if w in d else d["UNK"]
        X_train[i, idx] += 1

X_train = X_train.tocsr()

In [98]:
X_test = lil_matrix((len(test_df), len(d)), dtype=np.int32)

for i in range(len(test_df)):
    for w in test_df["text"].iloc[i].split():
        idx = d[w] if w in d else d["UNK"]
        X_test[i, idx] += 1

X_test = X_test.tocsr()

In [99]:
X_train
X_test

<Compressed Sparse Row sparse matrix of dtype 'int32'
	with 2525861 stored elements and shape (70000, 119965)>

In [100]:
y_train = train_df["Class Index"].values
y_test = test_df["Class Index"].values

In [101]:
y_train

array([ 0,  0,  0, ..., 13, 13, 13])

In [102]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print(classification_report(y_test, y_pred))

Accuracy: 0.9821
Macro F1: 0.9820995668886983
              precision    recall  f1-score   support

           0       0.95      0.95      0.95      5000
           1       0.98      0.98      0.98      5000
           2       0.96      0.97      0.96      5000
           3       0.99      0.99      0.99      5000
           4       0.98      0.98      0.98      5000
           5       0.99      0.99      0.99      5000
           6       0.97      0.97      0.97      5000
           7       0.99      0.99      0.99      5000
           8       0.99      1.00      0.99      5000
           9       0.99      0.99      0.99      5000
          10       0.99      0.99      0.99      5000
          11       0.99      0.99      0.99      5000
          12       0.98      0.98      0.98      5000
          13       0.97      0.97      0.97      5000

    accuracy                           0.98     70000
   macro avg       0.98      0.98      0.98     70000
weighted avg       0.98      0.98 

TF-IDF

In [103]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(min_df=5, max_df=0.8)

X_train = vectorizer.fit_transform(train_df["text"])
X_test = vectorizer.transform(test_df["text"])

In [104]:
y_train = train_df["Class Index"].values
y_test = test_df["Class Index"].values

In [105]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print(classification_report(y_test, y_pred))

Accuracy: 0.9830285714285715
Macro F1: 0.9830280669165405
              precision    recall  f1-score   support

           0       0.96      0.95      0.96      5000
           1       0.98      0.98      0.98      5000
           2       0.97      0.97      0.97      5000
           3       0.99      0.99      0.99      5000
           4       0.98      0.98      0.98      5000
           5       0.99      0.99      0.99      5000
           6       0.98      0.97      0.97      5000
           7       0.99      0.99      0.99      5000
           8       1.00      0.99      1.00      5000
           9       0.99      0.99      0.99      5000
          10       0.99      0.99      0.99      5000
          11       0.99      0.99      0.99      5000
          12       0.99      0.99      0.99      5000
          13       0.97      0.98      0.97      5000

    accuracy                           0.98     70000
   macro avg       0.98      0.98      0.98     70000
weighted avg       0.9

fastText

In [106]:
!pip install gensim

In [107]:
train_df[:5]

,Class Index,text
0,0,e d abbott ltd abbott of farnham e d abbott li...
1,0,stabilo stabilo is a german maker of pens for ...
2,0,q workshop q workshop is a polish company loca...
3,0,marvell software solutions israel marvell soft...
4,0,bergan mercy medical center bergan mercy medic...


In [108]:
text = " ".join(train_df["text"].tolist())
text = clean_text(text)
words = text.split()

word_counts = Counter(words)

# оставляем только слова, которые встречаются больше 5 раз
good_words = set(w for w in words if word_counts[w] > 5)

In [109]:
threshold = 1e-5
total_words = sum(word_counts[w] for w in good_words)

sentences = []

for text in train_df["text"]:
    text = clean_text(text)
    words = text.split()

    sentence = []

    for w in words:
        if w not in good_words:
            continue

        freq = word_counts[w] / total_words
        p_drop = 1 - (threshold / freq) ** 0.5

        if random.random() < p_drop:
            continue

        sentence.append(w)

    if len(sentence) > 0:
        sentences.append(sentence)

In [110]:
print(len(sentences))
print(sentences[0][:20])
print(sentences[1][:20])

557811
['abbott', 'ltd', 'abbott', 'farnham', 'abbott', 'limited', 'coachbuilding', 'farnham', 'surrey', 'trading', '1929', 'major', 'output', 'contract', 'vehicle', 'manufacturers', 'business']
['stabilo', 'stabilo', 'pens', 'colouring', 'and', 'cosmetics', 'markers', 'and', 'office', 'world', 'largest', 'pens', 'stabilo', 'boss']


In [111]:
from gensim.models import FastText
import numpy as np

fasttext_model = FastText(
    sentences=sentences,
    vector_size=128,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=5
)

In [112]:
def text_to_vector_fasttext(text, model):
    words = text.split()

    vectors = []
    for word in words:
        vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [113]:
X_train = np.array([
    text_to_vector_fasttext(text, fasttext_model)
    for text in train_df["text"]
])

X_test = np.array([
    text_to_vector_fasttext(text, fasttext_model)
    for text in test_df["text"]
])

y_train = train_df["Class Index"]
y_test = test_df["Class Index"]

In [114]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print(classification_report(y_test, y_pred))

Accuracy: 0.9406571428571429
Macro F1: 0.9406063048756655
              precision    recall  f1-score   support

           0       0.87      0.86      0.87      5000
           1       0.96      0.97      0.96      5000
           2       0.89      0.88      0.89      5000
           3       0.97      0.96      0.97      5000
           4       0.95      0.95      0.95      5000
           5       0.95      0.96      0.96      5000
           6       0.92      0.93      0.93      5000
           7       0.97      0.97      0.97      5000
           8       0.99      0.98      0.99      5000
           9       0.95      0.93      0.94      5000
          10       0.94      0.96      0.95      5000
          11       0.96      0.97      0.97      5000
          12       0.94      0.95      0.95      5000
          13       0.89      0.89      0.89      5000

    accuracy                           0.94     70000
   macro avg       0.94      0.94      0.94     70000
weighted avg       0.9